In [1]:
import json
import torch
from safetensors.torch import load_file
from transformers import PreTrainedTokenizerFast
from huggingface_hub import hf_hub_download
import tiktoken
from GPT2 import GPTModel

repo_id = "dinhxuanhuy/scratch_gpt2_tuned_interview"

config_path = hf_hub_download(repo_id, "config.json")
weight_path = hf_hub_download(repo_id, "model.safetensors")
tokenizer = tiktoken.get_encoding("gpt2")
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)


device = "cuda" if torch.cuda.is_available() else "cpu"

model = GPTModel(config)
state_dict = load_file(weight_path, device="cpu")

model.load_state_dict(state_dict, strict=True)
model.to(device)
model.eval()

print("Loaded successfully")

c:\Users\Transcend\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded successfully


In [3]:
def generate_text(
    model,
    idx,
    max_new_tokens,
    context_size,
    temperature=0.0,
    top_k=None,
    eos_id=50256
):
    model.eval()

    for _ in range(max_new_tokens):
        # chỉ lấy context cuối cùng nếu sequence dài hơn context_length
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        # lấy logits của token cuối
        logits = logits[:, -1, :]

        # top-k sampling nếu dùng
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1].unsqueeze(-1)
            logits = torch.where(
                logits < min_val,
                torch.tensor(float("-inf")).to(logits.device),
                logits
            )

        # greedy search
        if temperature == 0.0:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        # sampling với temperature
        else:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

        # dừng nếu gặp EOS
        if idx_next.item() == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [4]:
question = "What is machine learning?"

prompt = f"Question: {question}\nAnswer:"

input_ids = tokenizer.encode(prompt)
idx = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

out = generate_text(
    model=model,
    idx=idx,
    max_new_tokens=100,
    context_size=config["context_length"],
    temperature=0.0,   # 0.0 = greedy
    top_k=None,
    eos_id=50256
)

generated_text = tokenizer.decode(out[0].tolist())

print(generated_text)

Question: What is machine learning?
Answer: Yes, I use Python and PyTorch for PyTorch model development, with custom CUDA kernels to accelerate the rendering and optimization process.


In [14]:
question = "What is your GPA ?"

prompt = f"Question: {question}\nAnswer:"
idx = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)

out = generate_text(
    model=model,
    idx=idx,
    max_new_tokens=120,
    context_size=config["context_length"],
    temperature=0.0,
    eos_id=50256
)

text = tokenizer.decode(out[0].tolist())
answer = text[len(prompt):].strip()

print(answer)

To answer that, my GPA is 3.6 out of 4.0.
